In [1]:
import sys
sys.path.append("../src")

In [2]:
from datachecker.schema_loader import YamlFileSchemaLoader, SchemaSpec, SchemaLoader
from typing import Any, Type
from pydantic import BaseModel


In [4]:
loader: SchemaLoader = YamlFileSchemaLoader(root_dir="../schemas")
spec = loader.load("user")  # expects schemas/user.yaml
print(spec.name)            # "user"
print(spec.version)         # None unless you add "version:" under user:
print(spec.raw.keys())      # dict_keys(['schema', 'rules'])
print(spec.raw["schema"].keys())

user
None
dict_keys(['schema', 'rules'])
dict_keys(['id', 'first_name', 'last_name', 'email', 'created_at', 'is_active', 'country', 'system_x_id'])


In [5]:
spec

SchemaSpec(name='user', version=None, raw={'schema': {'id': {'type': 'integer', 'nullable': False, 'description': 'Unique user identifier (internal incremental id in generated data)'}, 'first_name': {'type': 'string', 'nullable': False, 'checks': {'str_length': {'min_value': 1, 'max_value': 100}}, 'description': 'User first name'}, 'last_name': {'type': 'string', 'nullable': False, 'checks': {'str_length': {'min_value': 1, 'max_value': 100}}, 'description': 'User last name'}, 'email': {'type': 'string', 'nullable': False, 'description': 'User email address'}, 'created_at': {'type': 'string', 'nullable': False, 'description': 'ISO-8601 datetime string when the record was created'}, 'is_active': {'type': 'boolean', 'nullable': False, 'description': 'Whether the user is active'}, 'country': {'type': 'string', 'nullable': False, 'description': 'ISO 3166-1 alpha-2 country code'}, 'system_x_id': {'type': 'string', 'nullable': False, 'description': 'External System X identifier (may contain l

In [6]:
from datachecker.plans import PydanticPlanCompiler, PydanticPlan

In [7]:
def validate_record(plan: PydanticPlan, record: dict[str, Any]) -> BaseModel:
    # returns validated object; raises pydantic.ValidationError if invalid
    return plan.model.model_validate(record)

In [8]:
plan = PydanticPlanCompiler().compile(spec)

In [9]:
user_obj = validate_record(plan, {"id": 11, "first_name": "Christopher", "last_name": "Williams", "email": "hernandezernest@example.net", "created_at": "1993-08-26T21:53:05.641380", "is_active": True, "country": "MV", "system_x_id": "400156"})

In [10]:
user_obj = validate_record(plan, {"id": 11, "first_name": "", "last_name": "Williams", "email": "hernandezernest@example.net", "created_at": "1993-08-26T21:53:05.641380", "is_active": True, "country": "MV", "system_x_id": "400156"})

ValidationError: 1 validation error for UserModel
first_name
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short